In [ ]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("student_data.csv")

In [ ]:
df.head()

In [ ]:
df["Passing Status"] = (df["G3"] >= 10).astype(int)

In [ ]:
df.drop(columns=["G3","G2"],axis = 1, inplace = True)

In [ ]:
X = df.drop(columns = ["Passing Status"], axis = 1)
y = df["Passing Status"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2, random_state=42, stratify=y)

In [ ]:
cat_cols = df.select_dtypes("object").columns.to_list()
transformer = ColumnTransformer(
    transformers = [
        ("onehot",OneHotEncoder(drop="first",handle_unknown="ignore"),cat_cols)
    ], remainder= "passthrough")

from sklearn.pipeline import Pipeline
pipe = Pipeline([("transformer", transformer),
                 ("scaler", StandardScaler()),
                 ("model",LogisticRegression(random_state=42, class_weight="balanced"))])

In [ ]:
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
score = accuracy_score(y_test,y_pred)
print(score)

In [ ]:
import mlflow
# Note: this path is specific to my machine. The Turkish letter "İ" in my user folder
# caused a file path error with SQLite, so I keep the MLflow database under C:/Users/Public.
# On another computer, replace it with your own path.
mlflow.set_tracking_uri("sqlite:///C:/Users/Public/mlflow.db")
mlflow.set_experiment("student-predictor")
with mlflow.start_run():
    C = 10
    pipe = Pipeline([("transformer", transformer),
                 ("scaler", StandardScaler()),
                 ("model",LogisticRegression(random_state=42, class_weight="balanced",C = C))])
    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    mlflow.log_param("C", C)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(pipe,name = "model")